# Notebook to Demonstrate NISAR Cal/Val Database GPS Access for Greenland Data
---

This demonstrates and tests the capability of the nisarSation class for working with GPS data processed by JPL for NISAR cryo cal/val activities.

## Python Setup

In [1]:
%load_ext autoreload
%autoreload 2
import time
#
reset = False
try:
    import nisarcryodb
except Exception:
    %pip install -e ~/./nisarcryodb
    reset = True
try:
    import nisargps
except Exception:
    %pip install -e ~/./nisargps
    reset = True
if reset:
    print('\n\033[1;31m\n\nRestart kernel and run this cell again \n\n \033[0m\n''')
    time.sleep(1e9) # stop from advancing

In [2]:
%load_ext autoreload
%autoreload 2
import numpy as np
import nisargps
import nisarcryodb
import matplotlib.pyplot as plt
from datetime import timedelta, datetime
import pyproj
from scipy.stats import linregress
import calendar
import os

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
def decimalYearToDateTime(date):
    if date is np.nan:
        return None
    year = int(date)
    fracYear = date - year
    yearLength = (datetime(year + 1, 1, 1) - datetime(year, 1, 1)).total_seconds() 
    return datetime(year, 1, 1) + timedelta(seconds=fracYear * yearLength)

## Load Data

Station data too big for github, so unzip if compressed after downloading.

Setup station id and path.

In [4]:
configFile = '../nisarcryodb/Notebooks/calvaldb_config.ini'

In [5]:
DB = nisarcryodb.nisarcryodb(configFile=configFile)

Error in: nisarcryodb._initDB
	Configuration file not found: ../nisarcryodb/Notebooks/calvaldb_config.ini


## Find Stations with Data

In [6]:
if False:  # set True to run this step
    d1 = 2018.0
    d2 = 2030.0
    stations = DB.getColumn('landice', 'gps_station', 'station_name')
    for station in stations:
        if 'test' not in station:
            myData = DB.getStationDateRangeData(station, d1, d2, 'landice', 'gps_data')
            minDate = decimalYearToDateTime(np.min(myData['decimal_year']) )
            maxDate = decimalYearToDateTime(np.max(myData['decimal_year']) )
            
            if minDate is not None:
                print(station, myData['station_id'][0], len(myData['station_id']), end='\t')
                print(minDate.strftime('%Y-%m-%d'), maxDate.strftime('%Y-%m-%d'))


## Setup NIT0

Pass in the already open DB rather than having the station open its own DB connection

In [7]:
%%time
stationName = 'NIT0'
stationNIT0 = nisargps.nisarStation(stationName, DBConnection=DB, traceBack=False)

Error in: nisarcryodb.stationNameToID
	'nisarcryodb' object has no attribute 'cursor'
None
Caching station data
station_id
Error in: nisarcryodb.getTableListing
	'nisarcryodb' object has no attribute 'cursor'
CPU times: user 280 μs, sys: 43 μs, total: 323 μs
Wall time: 307 μs


Load a sample data set. Can be accessed as `stationNIT0.x` (or `.y, .z, .lat, .lon, .date, .epoch`), where `date` provides the date as `datetime` array and `epoch` provides the date as a decimal year.

In [8]:
data = stationNIT0.subsetXYZ('2024-05-01 21:01:01', '2024-05-27 03:59:59', dateFormat='%Y-%m-%d %H:%M:%S', minPoints=1, traceBack=True)

Error in: nisarcryodb.stationNameToID
	'nisarcryodb' object has no attribute 'cursor'
Error in: nisarcryodb.getStationDateRangeData
	'nisarcryodb' object has no attribute 'cursor'

**************************************************
Error in: nisarStation._removeOverlap at line 760 
Message: 'NoneType' object is not subscriptable 
**************************************************
 


************************************************
Error in: nisarStation._subsetXYZDB at line 684 
Message: object of type 'NoneType' has no len() 
************************************************
 



SystemExit: 

/opt/conda/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Plot Positions

This function plots the relative trajectories of the GPS and computes the velocity two ways 1) `regression`, which uses the slope of the linear fit to the data, and 2) `point`, which uses the difference of the end points divided by time. For this latter method, you can control how many points at the end to average.

In [ ]:
def demoPlot(station, date1, date2):
    # Get points
    date, x, y, z, epoch = stationNIT0.subsetXYZ(date1, date2, sigmaMultiple=3)
    # Regression estimagte
    vx, vy, xMean, yMean = stationNIT0.computeVelocity(date1, date2, method='regression', dateFormat="%Y-%m-%d %H:%M:%S")
    # Get date extremes
    minDate = np.min(date).strftime('%Y-%m-%d')
    maxDate = np.max(date).strftime('%Y-%m-%d')
    # Do regression to demo (for plot only, not velocity estimate)
    resultx = linregress(epoch, x)
    resulty = linregress(epoch, y)
    # Compute linear path
    xline = resultx[0] * epoch + resultx[1]
    yline = resulty[0] * epoch + resulty[1]
    # Plot points
    fig, ax = plt.subplots(1, 1, figsize=(20, 10))
    ax.plot(epoch, x - xline, '.' )#, y - yline, 'r.', markersize=5, label='Regression Points')
    print(np.std(x - xline), np.std(y - yline))
    # Compute end point velocities
    #for ap, dy, color, markersize in zip([12, 3], [0.045, 0.09], ['black', 'blue'],[3, 3]):
        #continue
    #    dT = timedelta(days=ap)
    #    vxPt, vyPt, xMeanPt, yMeanPt = stationNIT0.computeVelocity(date1, date2, method='point', dateFormat="%Y-%m-%d %H:%M:%S", averagingPeriod=24*ap)
    #    ax.text(0.02, 0.03+dy, f'Point $\\pm${ap}days: vx={vxPt:.1f}, vy={vyPt:.1f}, v={np.sqrt(vxPt**2 + vyPt**2):.1f}',transform=ax.transAxes, fontsize=18)
   #    label = f'$\\pm${ap} end points'
     #   for myDate in [date1, date2]:
    #        dateX, x1, y1, z1, epoch1 = stationNIT0.subsetXYZ(myDate-dT, myDate+dT, sigmaMultiple=3)
     #       ax.plot(x1 - x[0], y1 - y[0], '.', color=color, markersize=markersize, label=label)
     #       label = None
    # Plot regression
    #ax.plot(xline-x[0], yline-y[0], 'orange', linewidth=2, label='Regression')
    # Decorate plot
    ax.set_xlabel('X (m)', fontsize=16)
    ax.set_ylabel('Y (m)', fontsize=16)
    ax.text(0.02, 0.03, f'Regression: vx={vx:.1f}, vy={vy:.1f}, v={np.sqrt(vx**2 + vy**2):.1f}',transform=ax.transAxes, fontsize=18)
    ax.legend()
    ax.set_title(f'Station {stationNIT0.stationName} Trajectory from {minDate} to {maxDate}', fontsize=14) 
    for lab  in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontsize(14)

Plot June data when there are continuous data.

In [ ]:
date1 = datetime(2025, 2, 11, 0, 0, 0)
date2 = datetime(2025, 2, 11, 23, 59, 59)

demoPlot(stationNIT0, date1, date2)

## Compute and Plot Time Series

This section demonstrates the estimate of a velocity time series over a given period. There are two methods:
- `point`, which computes the position averaged over some interval (`averagingPeriod`)
- `regression` this method computes the velocity of slope for all the points over the period of interest.

The method is specified with the `method` keyword. Either method generates the series from `date1` to `date2`, sampled at increments of `sampleInterval` hours. Velocities are computed with a `dT` specfied in hours. Further research is needed to best determine the velocity for NISAR cal/val. In this example, the standard deviations of the velocities computed using the two methods are 0.6 and 1.7 m/yr, respectively. The differences arise because of the large diurnal variation in speed on the floating ice. The results should be much better on grounded ice. 

The method and period are specified as:

In [ ]:
method = 'point'
#method = 'regression'
date1 = '2024-12-01' 
date2 = '2025-01-15'
averagingPeriod = 4
tides=True

Compute velocity at various intervals and over various periods.

Compute velocity time series at daily intervals.

In [ ]:
sampleInterval = 6 # Estimate every 4 hours
dT = 24  # Compute over a 24 hour interval
dateDaily, vxDaily, vyDaily, x, y = stationNIT0.computeVelocityTimeSeries(date1, date2,
                                                                    dT, sampleInterval, method=method, tides=tides, averagingPeriod=averagingPeriod)
print(f'Sigma X, Y {np.nanstd(vxDaily):.3} (m), {np.nanstd(vyDaily):.3} (m)')

Now compute at daily intervals but with delta T of 12 days.

In [ ]:
sampleInterval = 12 # Estimate every 12 hours
dT = 24 * 12  # Compute speed over 12 day interval
date12, vx12, vy12, x12, y12 = stationNIT0.computeVelocityTimeSeries(date1, date2,
                                                           dT, sampleInterval, method=method, tides=tides, averagingPeriod=averagingPeriod)
print(f'Sigma X, Y {np.nanstd(vx12):.3}, {np.nanstd(vy12):.3}')

Now Plot the results.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(18, 9))
for date, vx, vy, dTLabel, marker, linewidth in zip([ dateDaily, date12],
                                         [ vxDaily, vx12], 
                                         [ vyDaily, vy12],
                                         ['dT 1 day', 'dT 12 days'], 
                                         ['-', '-', 'o'], [.5, 1, 1]):
    ax.plot(date, vx, f'r{marker}', label=f'vx {dTLabel}', markersize=4, linewidth=linewidth)
    ax.plot(date, vy, f'b{marker}', label=f'vy {dTLabel}', markersize=4, linewidth=linewidth)
    ax.plot(date, np.sqrt(vx**2 + vy**2), f'k{marker}', label=f'$|v|$ {dTLabel}', markersize=4, linewidth=linewidth)
#
ax.set_ylabel('$v_x$, $v_y$, speed (m/yr', fontsize=18)  
ax.set_xlabel('Date', fontsize=18)
for lab  in ax.get_xticklabels() + ax.get_yticklabels():
    lab.set_fontsize(14)
ax.legend(loc='lower right', ncol=3)

As the results indicate, there is there is a large diurnal variation for the daily solutions, which are probably some kind of tide. It should be reduced with the new de-tided data from Ron. The 12-day estimates are quite stable with $\sigma$ of ~30 cm/yr. 

In [ ]:

print(d1, d2)

In [ ]:
dates = []
for year in range(2023, 2026):
    for month in range(1, 13):
        dates.append(datetime(year, month, 1))

In [ ]:
%%time
d1 = stationNIT0._datetimeToDecimalYear(datetime(2023,1,1))
d2 = stationNIT0._datetimeToDecimalYear(datetime(2026,1,1))
myData = stationNIT0.DB.getStationDateRangeData(stationNIT0.stationName, d1, d2,'landice', 'gps_data')



In [ ]:
%%time
for date1, date2 in zip(dates[0:-2], dates[1:-1]):
    x = myData[(myData["decimal_year"] > d1) & (myData["decimal_year"] < d2)]

        

In [ ]:
%%time
for date1, date2 in zip(dates[0:-2], dates[1:-1]):
    d1 = stationNIT0._datetimeToDecimalYear(date1)
    d2 = stationNIT0._datetimeToDecimalYear(date2)
    x =stationNIT0.DB.getStationDateRangeData(stationNIT0.stationName, d1, d2,
                                               'landice', 'gps_data')

In [ ]:
x

In [ ]:
mem_MB = myData.memory_usage(deep=True).sum() / (1024**2)
print(f"{mem_MB:.2f} MB")

In [ ]:
print(np.max(myData['version_id']))

In [ ]:
%%time
myID = stationNIT0.DB.stationNameToID('NIT0')
myData1 = stationNIT0.DB.getTableListing(schemaName='landice', tableName='gps_data', filters={'station_id': 5})

In [ ]:
stationNIT0.stationID

In [ ]:
myData1.station_id[1]

In [ ]:
d = {}
def test(d1):
    d1['x'] =1
test(d)
print(d)